# 04 · Audio — text-to-speech & transcription

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sabrinaaquino/base-batches-workshop/blob/main/notebooks/04-audio-tts-stt.ipynb)

Two endpoints:
- `/audio/speech` — text → audio (50+ voices, multilingual)
- `/audio/transcriptions` — audio → text (Whisper-style)

We'll do TTS first, then transcribe what we generated, then build a tiny **voice → idea → narrated commercial** pipeline.

In [ ]:
%pip install --quiet openai requests python-dotenv rich

In [ ]:
import os, sys

# Try Colab secrets first
try:
    from google.colab import userdata  # type: ignore
    api_key = userdata.get("VENICE_API_KEY")
    os.environ["VENICE_API_KEY"] = api_key
except Exception:
    api_key = os.environ.get("VENICE_API_KEY")

if not api_key:
    from getpass import getpass
    api_key = getpass("Paste your Venice API key: ").strip()
    os.environ["VENICE_API_KEY"] = api_key

from openai import OpenAI
client = OpenAI(
    api_key=api_key,
    base_url="https://api.venice.ai/api/v1",
)
print("Connected to Venice ✔")

## 1. Text to speech

The `/audio/speech` endpoint is OpenAI-compatible. Returns binary audio.

In [ ]:
from IPython.display import Audio, display

speech = client.audio.speech.create(
    model="kokoro",
    voice="af_heart",
    input="Welcome to Base Batches. Today we're shipping AI you can actually trust.",
    response_format="mp3",
)

with open("welcome.mp3", "wb") as f:
    f.write(speech.content)

display(Audio("welcome.mp3"))

## 2. List available voices

Voices vary by model. Kokoro alone has dozens — different genders, accents, languages.

In [ ]:
import requests

models = requests.get(
    "https://api.venice.ai/api/v1/models",
    headers={"Authorization": f"Bearer {api_key}"},
    params={"type": "tts"},
).json()

for m in models.get("data", []):
    spec = m.get("model_spec", {})
    voices = (spec.get("capabilities", {}) or {}).get("voices", [])
    if voices:
        print(f"{m['id']}: {len(voices)} voices")
        print(f"   sample: {', '.join(voices[:6])}{'...' if len(voices) > 6 else ''}")

## 3. Speech to text — round-trip what we just made

In [ ]:
with open("welcome.mp3", "rb") as f:
    transcript = client.audio.transcriptions.create(
        model="whisper-large-v3",
        file=f,
    )

print("Transcript:")
print(transcript.text)

## 4. The full pipeline: voice memo → script → narrated ad

Real-world flow that uses three Venice endpoints in 30 lines.

In [ ]:
# Step 1: Pretend we already recorded a voice memo. To keep this notebook
# self-contained, we generate one with TTS first.
memo_text = "My idea is a vending machine that sells regret."
memo_audio = client.audio.speech.create(
    model="kokoro", voice="af_heart", input=memo_text, response_format="mp3"
).content
with open("idea.mp3", "wb") as f:
    f.write(memo_audio)

# Step 2: Transcribe (in production this would be the user's actual recording)
with open("idea.mp3", "rb") as f:
    idea = client.audio.transcriptions.create(model="whisper-large-v3", file=f).text
print(f"💡 Transcribed idea: {idea}\n")

# Step 3: Have a model write a 4-second commercial script
script = client.chat.completions.create(
    model="venice-uncensored",
    messages=[
        {"role": "system", "content": "You are a darkly funny ad copywriter. Output ONLY the spoken voiceover text — no stage directions, no quotes, no preamble. Maximum 25 words."},
        {"role": "user", "content": f"Idea: {idea}"},
    ],
).choices[0].message.content.strip()
print(f"📝 Script: {script}\n")

# Step 4: Narrate the script
ad = client.audio.speech.create(
    model="kokoro", voice="am_michael", input=script, response_format="mp3"
).content
with open("ad.mp3", "wb") as f:
    f.write(ad)

print("🎬 Listen:")
display(Audio("ad.mp3"))

## What just happened

You went from a voice memo → text → creative script → audio commercial in one notebook, hitting **three** Venice endpoints with one auth.

This is the entire stack you'd otherwise piece together from OpenAI, ElevenLabs, and Whisper.cpp.

**Next:** [05 · Video](./05-video-generation.ipynb)